# Reentrenar YOLOv8 Nano para detectar jaguares

Este notebook descarga tu dataset de Roboflow, entrena un modelo YOLOv8 nano y genera un archivo `best.pt` descargable para integrarlo luego en el backend.

## 1. Activar GPU en Colab

En Colab entra a `Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > GPU` antes de correr el entrenamiento.

In [ ]:
!nvidia-smi

## 2. Instalar dependencias

Ultralytics usa PyTorch por debajo para entrenar YOLOv8.

In [ ]:
!pip install -q ultralytics roboflow

import torch
from ultralytics import YOLO

print('PyTorch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 3. Descargar dataset desde Roboflow

Usa tu workspace `jhow-rambo`, proyecto `jaguar`, version `4`, formato YOLOv8.

In [ ]:
from roboflow import Roboflow
from pathlib import Path

ROBOFLOW_API_KEY = 'SoIQZPPcmUe4QxZ69IjM'

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('jhow-rambo').project('jaguar')
version = project.version(4)
dataset = version.download('yolov8')

dataset_dir = Path(dataset.location)
data_yaml = dataset_dir / 'data.yaml'

print('Dataset:', dataset_dir)
print('data.yaml:', data_yaml)
!ls -lah {dataset_dir}

## 4. Revisar configuracion del dataset

El archivo `data.yaml` debe apuntar a train/valid/test y contener la clase jaguar.

In [ ]:
print(data_yaml.read_text())

## 5. Entrenar YOLOv8 nano

El modelo base es `yolov8n.pt`, la version nano. Puedes subir `epochs` si tienes mas tiempo/GPU.

In [ ]:
model = YOLO('yolov8n.pt')

results = model.train(
    data=str(data_yaml),
    epochs=80,
    imgsz=640,
    batch=-1,
    patience=20,
    optimizer='auto',
    device=0 if torch.cuda.is_available() else 'cpu',
    project='jaguar_training',
    name='yolov8n_jaguar',
    pretrained=True,
    plots=True
)

## 6. Validar el mejor modelo

In [ ]:
best_model_path = Path('jaguar_training/yolov8n_jaguar/weights/best.pt')
last_model_path = Path('jaguar_training/yolov8n_jaguar/weights/last.pt')

print('best.pt existe:', best_model_path.exists(), best_model_path)
print('last.pt existe:', last_model_path.exists(), last_model_path)

best_model = YOLO(str(best_model_path))
metrics = best_model.val(data=str(data_yaml), imgsz=640, device=0 if torch.cuda.is_available() else 'cpu')
metrics

## 7. Probar inferencia con imagenes de validacion

In [ ]:
valid_images = list((dataset_dir / 'valid' / 'images').glob('*'))
print('Imagenes validacion:', len(valid_images))

if valid_images:
    sample_images = valid_images[:8]
    pred_results = best_model.predict(
        source=[str(p) for p in sample_images],
        imgsz=640,
        conf=0.25,
        save=True,
        project='jaguar_training',
        name='predicciones_validacion'
    )
    print('Predicciones guardadas en jaguar_training/predicciones_validacion')

## 8. Exportar modelo

Para el backend de este proyecto basta con `best.pt`. Tambien se exporta ONNX por si luego quieres usarlo fuera de PyTorch.

In [ ]:
exported_onnx = best_model.export(format='onnx', imgsz=640)
print('Modelo PyTorch:', best_model_path)
print('Modelo ONNX:', exported_onnx)

## 9. Comprimir y descargar resultados

Descarga el ZIP. Para integrar al backend usaremos principalmente `best.pt`.

In [ ]:
!mkdir -p export_model
!cp jaguar_training/yolov8n_jaguar/weights/best.pt export_model/yolov8n_jaguar_best.pt
!cp jaguar_training/yolov8n_jaguar/weights/last.pt export_model/yolov8n_jaguar_last.pt
!cp jaguar_training/yolov8n_jaguar/results.png export_model/results.png || true
!cp jaguar_training/yolov8n_jaguar/confusion_matrix.png export_model/confusion_matrix.png || true
!cp jaguar_training/yolov8n_jaguar/PR_curve.png export_model/PR_curve.png || true
!zip -r yolov8n_jaguar_model.zip export_model

from google.colab import files
files.download('yolov8n_jaguar_model.zip')

## 10. Guardar en Google Drive opcional

Ejecuta esta celda si quieres dejar una copia permanente en tu Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/modelos_jaguar
!cp yolov8n_jaguar_model.zip /content/drive/MyDrive/modelos_jaguar/
!cp export_model/yolov8n_jaguar_best.pt /content/drive/MyDrive/modelos_jaguar/

print('Guardado en /content/drive/MyDrive/modelos_jaguar/')

## 11. Como integrarlo luego en el backend

Cuando tengas `yolov8n_jaguar_best.pt`, colocalo en el proyecto, por ejemplo en:

`app/utils/models/yolov8n_jaguar_best.pt`

Luego ajustaremos el detector para cargar ese modelo y mapear su clase `jaguar` directamente.